In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv('tccs.csv', sep=';')

In [4]:
df.head()

,titulo,palavras_chave,nome_autor,nome_orientador,nome_orientador_externo,curso,nivel,tipo_trabalho,ano,data_defesa
0,"""A definir""","cadeia de custódia, processo penal, prova.",BRUNO LUIS CASTRO DA SILVA,WALTER NUNES DA SILVA JUNIOR,NaN,DIREITO,GRADUAÇÃO,MONOGRAFIA,2022.0,6202/09/02 00:00:00.000
1,MÚLTIPLA FILIAÇÃO REGISTRAL: ANÁLISE CONSTITUC...,NÃO INFORMADO.,LORENA NOGUEIRA RÊGO,ERICA VERICIA CANUTO DE OLIVEIRA VERAS,NaN,DIREITO,GRADUAÇÃO,MONOGRAFIA,2015.0,5015/11/25 00:00:00.000
2,O Impacto da Lei de Responsabilidade Fiscal so...,NaN,JULIANA CRISTINA WERNECK,ZIVANILSON TEIXEIRA E SILVA,NaN,CIÊNCIAS ECONÔMICAS,GRADUAÇÃO,MONOGRAFIA,2013.0,3013/12/17 00:00:00.000
3,Avaliação do Programa Agroamigo dos Território...,NaN,KALBY ANDSON ELOI LEITE,ZIVANILSON TEIXEIRA E SILVA,NaN,CIÊNCIAS ECONÔMICAS,GRADUAÇÃO,MONOGRAFIA,2013.0,3013/12/17 00:00:00.000
4,A EXIGIBILIDADE DA VACINAÇÃO COMO MEDIDA PROFI...,Vacinação; COVID-19; Obrigatoriedade; Atletas ...,DIEGO ALBERTO FARIAS DANTAS,DIOGO PIGNATARO DE OLIVEIRA,NaN,DIREITO,GRADUAÇÃO,MONOGRAFIA,2022.0,2202/12/21 00:00:00.000


In [5]:
# Pegando colunas que tem valores do tipo string

colunas_string = df.select_dtypes(include=['object', 'string']).columns

In [6]:
# transformando as colunas string em letras minúsculas

df[colunas_string] = df[colunas_string].apply(lambda x: x.str.lower())

In [7]:
# todos os títulos que contêm 'a definir' escrito de maneiras diferentes

df[df['titulo'].str.contains("a definir", case=False, na=False)]

,titulo,palavras_chave,nome_autor,nome_orientador,nome_orientador_externo,curso,nivel,tipo_trabalho,ano,data_defesa
0,"""a definir""","cadeia de custódia, processo penal, prova.",bruno luis castro da silva,walter nunes da silva junior,NaN,direito,graduação,monografia,2022.0,6202/09/02 00:00:00.000
17142,"""a definir""",sanções econômicas; evasão; análise econômica ...,bruno henrique macedo de medeiros,otacilio dos santos silveira neto,NaN,direito,graduação,monografia,2022.0,2022/07/28 00:00:00.000
21498,a definir,"public transparency, accountability, state-own...",josé luiz moreira rebouças,otacilio dos santos silveira neto,NaN,direito,graduação,monografia,2020.0,2021/12/31 00:00:00.000
21502,a definir,"transparência pública, accountability, lei das...",josé luiz moreira rebouças,otacilio dos santos silveira neto,NaN,direito,graduação,monografia,2020.0,2021/12/31 00:00:00.000
31693,a definir,NaN,fernando henrique targino,yara maria pereira gurgel,NaN,direito,graduação,monografia,2020.0,2020/04/20 00:00:00.000
31694,a definir,NaN,josé de anchieta cruz neto,yara maria pereira gurgel,NaN,direito,graduação,monografia,2020.0,2020/04/20 00:00:00.000
31695,a definir,NaN,carlos andré correia lima moreno,yara maria pereira gurgel,NaN,direito,graduação,monografia,2020.0,2020/04/20 00:00:00.000
31700,a definir,palavras-chave: direito fundamental à saúde me...,camilla amanda aires de medeiros,yara maria pereira gurgel,NaN,direito,graduação,monografia,2020.0,2020/04/20 00:00:00.000
31701,ainda a definir,labor compliance. prevention. labor actions. m...,steffisson oliveira da cruz,yara maria pereira gurgel,NaN,direito,graduação,monografia,2020.0,2020/04/20 00:00:00.000
55617,a definir,NaN,josé wilson da silva santos,hipolito virgilio magalhaes junior,NaN,fonoaudiologia,graduação,monografia,2015.0,2016/06/06 00:00:00.000


In [8]:
# essas duas linhas possuem a definir no título mas são válidas (verificado manualmente)

titulos_adefinir_validos = df.iloc[[69265, 69291]]

In [9]:
indices_invalidos = df[df['titulo'].str.contains("a definir", case=False, na=True)].index

In [10]:
df = df.drop(indices_invalidos)

In [11]:
# adicionando os títulos armazenados em titulos_adefinir_validos novamente no data frame

df = pd.concat([df, titulos_adefinir_validos])

In [12]:
# outro tipo de título inválido ("não informado")

indices_invalidos2 = df[df['titulo'].str.contains("não informado", case=False, na=True)].index

In [13]:
df = df.drop(indices_invalidos2)

In [14]:
# decidi dropar a coluna palavras chave já que é uma informação que já é parcialmente trazida pela coluna de título e tem 53% dos valores ausentes além daqueles que estão como "a definir", "nao informado" ou "nao consta"

df = df.drop('palavras_chave', axis=1)

In [15]:
# substitui strings vazias (apenas espaços) por nan
df['nome_orientador'] = df['nome_orientador'].replace(r'^\s*$', np.nan, regex=True) 
#verificar se realmente é últil

In [16]:
# Verificar cursos onde todos os orientadores são nulos
cursos_todos_nulos = (
    df.groupby('curso')['nome_orientador']
      .apply(lambda x: x.isna().all())
)

# Mostrar apenas os cursos com todos os valores nulos
cursos_todos_nulos = cursos_todos_nulos[cursos_todos_nulos]
print("Cursos com todos os valores nulos em 'nome_orientador':")
print(cursos_todos_nulos.index.tolist())


Cursos com todos os valores nulos em 'nome_orientador':
['especialização em desenvolvimento de sistemas ufrn', 'especialização em ensino de filosofia no ensino médio']


In [17]:
# Preenche o nome orientador nulo pela media do curso. Em casos que todos os os prof orientadores do curso são nulos, preenche pela moda geral. Em uma pesquisa breve, não achei nada que indicasse nesses dois cursos (especialização em desenvolvimento de sistemas ufrn', 'especialização em ensino de filosofia no ensino médio) que não precisa de professor orientador, logo acredito ser um erro e decidi tratar

def preencher_por_moda(grupo):
    moda = grupo['nome_orientador'].mode()
    moda_geral = df['nome_orientador'].mode()
    if not moda.empty:
        grupo['nome_orientador'] = grupo['nome_orientador'].fillna(moda[0])
    else:
        grupo['nome_orientador'] = grupo['nome_orientador'].fillna(moda_geral[0])
    return grupo

In [18]:
df = df.groupby('curso', group_keys=False).apply(preencher_por_moda)

C:\Users\pedro\AppData\Local\Temp\ipykernel_15892\4266047769.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('curso', group_keys=False).apply(preencher_por_moda)


In [19]:
print(df['nome_orientador'].isna().sum())

0


In [20]:
# coluna com muitos valores ausentes

df = df.drop('nome_orientador_externo', axis=1)

In [21]:
df['tipo_trabalho'].unique()

array(['monografia', 'artigo científico', 'outros', nan, 'memorial',
       'prática', 'seminário'], dtype=object)

In [22]:
df['nivel'].unique()

array(['graduação', 'lato sensu', 'doutorado', 'mestrado',
       'stricto sensu'], dtype=object)

In [23]:
# conta quantos valores de tipo trabalho existem para cada nível

df_com_nan = df[df['tipo_trabalho'].isna()]

contagem = df_com_nan['nivel'].value_counts()

print(contagem)

nivel
mestrado         13307
doutorado         4540
stricto sensu        1
Name: count, dtype: int64


In [24]:
df['nivel'].value_counts()

nivel
graduação        49715
lato sensu       17330
mestrado         13307
doutorado         4540
stricto sensu        1
Name: count, dtype: int64

In [25]:
#não há valores nulos de tipo trabalho para graduação e para todos os outros níveis tipo trabalho é nan, logo acredito que os valores nulos nessa coluna não são erros

In [26]:
df.sort_values(by='ano', ascending=False)

,titulo,nome_autor,nome_orientador,curso,nivel,tipo_trabalho,ano,data_defesa
70242,levantamento epidemiológico descritivo das pri...,ana paula rufino santos da costa,marcilio dias chaves de oliveira,odontologia,graduação,artigo científico,3013.0,2013/12/04 00:00:00.000
35488,"funções executivas, memória e aprendizagem no ...",nirlena carla pereira dantas da silva,janaina weissheimer,especialização em alfabetização e neurociência...,lato sensu,monografia,2091.0,2019/07/12 00:00:00.000
2112,o ensino de biomas nos anos finais do ensino f...,ana katarina nascimento de azevedo,luiz alberto da silva junior,especialização em ensino de ciências - anos fi...,lato sensu,monografia,2025.0,2025/01/27 00:00:00.000
56,reflexões sobre a conciliação trabalhista: est...,ressu ferreira pires,luciano athayde chaves,mestrado em direito,mestrado,NaN,2025.0,2025/07/30 00:00:00.000
21,indicadores de desempenho: um estudo da perspe...,alvaro araujo de medeiros,marconi neves macedo,administração,graduação,monografia,2025.0,2025/09/09 00:00:00.000
...,...,...,...,...,...,...,...,...
86824,exportação da lagosta no estado do rio grande ...,katia maria dias carmo ribeiro,iloneide carlos de oliveira ramos,estatística,graduação,monografia,1988.0,1989/01/11 00:00:00.000
85140,tributação no brasil,mauricea bezerra de araujo,josue lins e silva,ciências contábeis,graduação,outros,208.0,2008/07/01 00:00:00.000
22539,ações para a melhoria da cobertura vacinal de ...,israela santos barbalho,daniele vieira dantas,especialização em saúde da família,lato sensu,monografia,202.0,2021/10/14 00:00:00.000
35693,integração do planejamento governamental com o...,andré luis alves de quevedo,jovanka bittencourt leite de carvalho,especialização em gestão do trabalho e da educ...,lato sensu,outros,2.0,2019/07/05 00:00:00.000


In [27]:
# substituindo outliers na coluna ano verificando a coluna 'data_defesa'

df['ano'] = df['ano'].replace(3013, 2013)

In [28]:
df['ano'] = df['ano'].replace(2091, 2019)

In [29]:
df.sort_values(by='ano', ascending=True)

,titulo,nome_autor,nome_orientador,curso,nivel,tipo_trabalho,ano,data_defesa
35140,perfil neuropsicológico em criança com má form...,raquel maria de souza lima queiroz,izabel augusta hazin pires,especialização em neuropsicologia clínica,lato sensu,prática,1.0,2019/07/31 00:00:00.000
35693,integração do planejamento governamental com o...,andré luis alves de quevedo,jovanka bittencourt leite de carvalho,especialização em gestão do trabalho e da educ...,lato sensu,outros,2.0,2019/07/05 00:00:00.000
22539,ações para a melhoria da cobertura vacinal de ...,israela santos barbalho,daniele vieira dantas,especialização em saúde da família,lato sensu,monografia,202.0,2021/10/14 00:00:00.000
85140,tributação no brasil,mauricea bezerra de araujo,josue lins e silva,ciências contábeis,graduação,outros,208.0,2008/07/01 00:00:00.000
86824,exportação da lagosta no estado do rio grande ...,katia maria dias carmo ribeiro,iloneide carlos de oliveira ramos,estatística,graduação,monografia,1988.0,1989/01/11 00:00:00.000
...,...,...,...,...,...,...,...,...
71,algoritmos quasi-newton: desempenho dos método...,hilário petronio de medeiros dantas,francisco marcio barboza,sistemas de informação,graduação,monografia,2025.0,2025/07/25 00:00:00.000
53,"estética, satisfação, qualidade de vida e dese...",liliane cristina nogueira marinho,patricia dos santos calderon,doutorado em ciências odontológicas,doutorado,NaN,2025.0,2025/08/01 00:00:00.000
54,formação didático-pedagógica em robótica educa...,italo pereira de melo,aquiles medeiros filgueira burlamaqui,mestrado profissional em inovação em tecnologi...,mestrado,NaN,2025.0,2025/08/01 00:00:00.000
57,corpo e intencionalidade: análise da crítica d...,vinicius alves bastos medeiros,paulo eduardo bodziak junior,mestrado em filosofia,mestrado,NaN,2025.0,2025/07/29 00:00:00.000


In [30]:
df['ano'] = df['ano'].replace(1, 2019)

In [31]:
df['ano'] = df['ano'].replace(2, 2019)

In [32]:
df['ano'] = df['ano'].replace(202, 2021)

In [33]:
df['ano'] = df['ano'].replace(208, 2008)

In [34]:
df.to_csv(
    'tccs_tratado.csv',
    sep=';',
    encoding='utf-8',
    index=False
)